In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, StratifiedKFold,GridSearchCV
from sklearn.metrics import mean_squared_error, make_scorer, r2_score, confusion_matrix
import numpy as np
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFECV
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [ ]:
df = pd.read_excel('/content/all_data_0710.xlsx')

# drop all rows with nan
df=df.dropna(axis=0)

In [ ]:
selected_columns=['accident type', '发生地种类', 'season', 'wind speed level', 'dayNight',
       'fatality', '作业情况', '损伤种类', '损伤位置', 'wind direction', '能见度等级', '是否超速',
       '路况种类', 'ship type', '船体材料', '主机类型', '船长', '船宽', '设备故障', '环境恶劣']

In [ ]:
def filter_rare_values(df, selected_cols, threshold=5):
    for col in selected_columns:
        value_counts = df[col].value_counts()
        df = df[df[col].map(value_counts) >= threshold]  # Keep rows where the value appears ≥ threshold
    return df
df=filter_rare_values(df,selected_columns)
df.drop(df[df['accident type'] == 'Others'].index, inplace=True)

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
model = BertModel.from_pretrained("bert-base-chinese")

def get_embedding(text):
    # Tokenize the text
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use the mean pooling of the last hidden state for sentence-level embeddings
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

In [ ]:
Cause_Embeddings = df['Cause of Accident'].fillna("").apply(lambda x: get_embedding(x))
Traffic_Embeddings=df['事发水域路况'].fillna("").apply(lambda x: get_embedding(x))

df['Bert_cause'],df['Bert_traffic']=Cause_Embeddings,Traffic_Embeddings

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=1000)

def get_tfidf_embedding(text):
    # Vectorize the text using the fitted TF-IDF vectorizer
    embedding = vectorizer.transform([text]).toarray().squeeze()
    return embedding

vectorizer.fit(df['Cause of Accident'].fillna(""))
# Apply the TF-IDF embedding function to each row in the DataFrame column
Cause_Embeddings = df['Cause of Accident'].fillna("").apply(lambda x: get_tfidf_embedding(x))


vectorizer.fit(df['事发水域路况'].fillna(""))
Traffic_Embeddings= df['事发水域路况'].fillna("").apply(lambda x: get_tfidf_embedding(x))

df['Tfidf_cause'],df['Tfidf_traffic']=Cause_Embeddings,Traffic_Embeddings

In [ ]:
df = df.drop(['Cause of Accident', '事发水域路况'], axis=1)

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

class TextEmbeddingClusterModel:
    def __init__(self, n_clusters=7, seed=1):
        self.n_clusters = n_clusters
        self.seed = seed

        self.kmeans_cause_bert = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)
        self.kmeans_traffic_bert = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)
        self.kmeans_cause_tfidf = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)
        self.kmeans_traffic_tfidf = KMeans(n_clusters=n_clusters, random_state=seed,n_init=10)

    def train(self, df):
        # Expecting these columns to contain array-like vectors
        if 'Bert_cause' in df.columns:
            cause_bert_matrix = np.vstack(df['Bert_cause'].values)
            self.kmeans_cause_bert.fit(cause_bert_matrix)
        if 'Bert_traffic' in df.columns:
            traffic_bert_matrix = np.vstack(df['Bert_traffic'].values)
            self.kmeans_traffic_bert.fit(traffic_bert_matrix)

        if 'Tfidf_cause' in df.columns:
            cause_tfidf_matrix = np.vstack(df['Tfidf_cause'].values)
            self.kmeans_cause_tfidf.fit(cause_tfidf_matrix)

        if 'Tfidf_traffic' in df.columns:
            traffic_tfidf_matrix = np.vstack(df['Tfidf_traffic'].values)
            self.kmeans_traffic_tfidf.fit(traffic_tfidf_matrix)


        return self

    def predict(self, df):
        cause_bert_clusters = None
        traffic_bert_clusters = None
        cause_tfidf_clusters = None
        traffic_tfidf_clusters = None

        if 'Bert_cause' in df.columns:
            cause_bert_matrix = np.vstack(df['Bert_cause'].values)
            cause_bert_clusters = self.kmeans_cause_bert.predict(cause_bert_matrix)

        if 'Bert_traffic' in df.columns:
            traffic_bert_matrix = np.vstack(df['Bert_traffic'].values)
            traffic_bert_clusters = self.kmeans_traffic_bert.predict(traffic_bert_matrix)

        if 'Tfidf_cause' in df.columns:
            cause_tfidf_matrix = np.vstack(df['Tfidf_cause'].values)
            cause_tfidf_clusters = self.kmeans_cause_tfidf.predict(cause_tfidf_matrix)

        if 'Tfidf_traffic' in df.columns:
            traffic_tfidf_matrix = np.vstack(df['Tfidf_traffic'].values)
            traffic_tfidf_clusters = self.kmeans_traffic_tfidf.predict(traffic_tfidf_matrix)

        return cause_tfidf_clusters, traffic_tfidf_clusters, cause_bert_clusters, traffic_bert_clusters



In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import f1_score  # Or other method

def run_cv_round(X_train, y_train, models, skf, seed, use_feature_selection=False,scoring='accruacy'):
    SCORING_FUNCTIONS= {
      'accuracy': accuracy_score,
      'f1_macro': lambda y_true, y_pred: f1_score(y_true, y_pred, average='macro'),
      'recall_macro': lambda y_true, y_pred: recall_score(y_true, y_pred, average='macro'),
  }
    best_model_name = None
    best_avg_score = -np.inf
    best_params = None
    best_model_class = None

    for name, (model, param_grid) in models.items():
        print(f"Training model: {name} | Feature Selection: {use_feature_selection}")
        fold_accuracies = []
        best_param_candidates = []

        for train_idx, test_idx in skf.split(X_train, y_train):
            fold_trainx, fold_trainy = X_train.iloc[train_idx].copy(), y_train.iloc[train_idx].copy()
            fold_testx, fold_testy = X_train.iloc[test_idx].copy(), y_train.iloc[test_idx].copy()

            # Embedding as before
            embedding_model = TextEmbeddingClusterModel(n_clusters=7, seed=seed)
            embedding_model.train(fold_trainx)
            for df, name in [(fold_trainx, "train"), (fold_testx, "test")]:
                try:
                    tfidf_cause, tfidf_traffic, bert_cause, bert_traffic = embedding_model.predict(df)
                    if tfidf_cause is not None: df['Tfidf_cause'] = tfidf_cause
                    if tfidf_traffic is not None: df['Tfidf_traffic'] = tfidf_traffic
                    if bert_cause is not None: df['Bert_cause'] = bert_cause
                    if bert_traffic is not None: df['Bert_traffic'] = bert_traffic
                except Exception as e:
                    print(f"Warning during {name} embedding on fold: {e}")

            # 🟡 Apply feature selection (optional)
            if use_feature_selection:
                selector = SelectKBest(score_func=f_classif, k='all')  # or choose K
                selector.fit(fold_trainx, fold_trainy)
                fold_trainx = pd.DataFrame(selector.transform(fold_trainx), columns=fold_trainx.columns[selector.get_support()])
                fold_testx = pd.DataFrame(selector.transform(fold_testx), columns=fold_trainx.columns)

            # Grid search
            try:
                grid_search = GridSearchCV(model, param_grid=param_grid, cv=3, scoring=scoring)
                grid_search.fit(fold_trainx, fold_trainy)
                preds = grid_search.best_estimator_.predict(fold_testx)
                metric_func = SCORING_FUNCTIONS.get(scoring, accuracy_score)
                score = metric_func(fold_testy, preds)
                fold_accuracies.append(score)
                best_param_candidates.append(grid_search.best_params_)
            except Exception as e:
                print(f"Grid search or prediction failed: {e}")

        avg_score = np.mean(fold_accuracies)
        print(f"{name} average CV score: {avg_score:.4f} | Feature Selection: {use_feature_selection}")

        if avg_score > best_avg_score:
            best_avg_score = avg_score
            best_model_name = name
            best_model_class = model.__class__
            best_index = np.argmax(fold_accuracies)
            best_params = best_param_candidates[best_index]

    return best_model_name, best_model_class, best_params, best_avg_score


In [ ]:
def cross_validate(seed, X_train, y_train, scoring='accuracy'):
    # models = {
    #     "GBM": (
    #         GradientBoostingClassifier(),
    #         {
    #             'n_estimators': [100, 150, 200],
    #             'learning_rate': [0.03, 0.05, 0.1],
    #             'max_depth': [3, 5, 7],
    #             'min_samples_split': [2, 4, 6],
    #             'min_samples_leaf': [1, 2],
    #             'subsample': [0.8, 1.0],
    #             'max_features': ['sqrt', 'log2']
    #         }
    #     ),
    #     "RF": (
    #         RandomForestClassifier(),
    #         {
    #             'n_estimators': [100, 150, 200],
    #             'max_depth': [None, 15, 25],
    #             'min_samples_split': [2, 4, 6],
    #             'min_samples_leaf': [1, 2],
    #             'bootstrap': [True],
    #             'criterion': ['gini'],
    #             'max_features': ['sqrt', 'log2'],
    #             'max_leaf_nodes': [None, 20, 30]
    #         }
    #     )
    # }
    models = {
        "GBM": (
            GradientBoostingClassifier(),
            {
                'n_estimators': [100],             # Fixed
                'learning_rate': [0.1],            # Fixed
                'max_depth': [3],                  # Simpler tree
            }
        ),
        "RF": (
            RandomForestClassifier(),
            {
                'n_estimators': [100],             # Fixed
                'max_depth': [None],               # Default depth
                'max_features': ['sqrt']           # Good default
            }
        )
    }
    skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)

    r1_name, r1_class, r1_params, r1_score = run_cv_round(X_train, y_train, models, skf, seed, use_feature_selection=False, scoring=scoring)
    r2_name, r2_class, r2_params, r2_score = run_cv_round(X_train, y_train, models, skf, seed, use_feature_selection=True, scoring=scoring)

    if r1_score >= r2_score:
        final_model_class = r1_class
        final_params = r1_params
        final_name = r1_name
        use_feature_selection = False
        print(f"Selected model WITHOUT feature selection: {final_name} | Score: {r1_score:.4f}")
    else:
        final_model_class = r2_class
        final_params = r2_params
        final_name = r2_name
        use_feature_selection = True
        print(f"Selected model WITH feature selection: {final_name} | Score: {r2_score:.4f}")

    # Final training phase
    final_model = final_model_class(**final_params)

    embedding_model = TextEmbeddingClusterModel(n_clusters=7, seed=seed)
    embedding_model.train(X_train)
    try:
        tfidf_cause, tfidf_traffic, bert_cause, bert_traffic = embedding_model.predict(X_train)
        if tfidf_cause is not None: X_train['Tfidf_cause'] = tfidf_cause
        if tfidf_traffic is not None: X_train['Tfidf_traffic'] = tfidf_traffic
        if bert_cause is not None: X_train['Bert_cause'] = bert_cause
        if bert_traffic is not None: X_train['Bert_traffic'] = bert_traffic
    except Exception as e:
        print(f"Embedding failed on training data: {e}")

    if use_feature_selection:
        selector = SelectKBest(score_func=f_classif, k='all')
        selector.fit(X_train, y_train)
        selected_columns = X_train.columns[selector.get_support()]
        X_train = pd.DataFrame(selector.transform(X_train), columns=selected_columns)
        final_model.fit(X_train, y_train)
        return selector,final_model, max(r1_score, r2_score)
    else:
      final_model.fit(X_train,y_train)
      return None,final_model,max(r1_score,r2_score)

In [ ]:
def type_prediction(df, seed_list, output_csv="9_11_1a_allmethod.csv"):
    results = []

    for seed in seed_list:
        print(f"Started seed {seed}")

        # 1. Split the data
        df_train, df_test = train_test_split(df, test_size=0.3, random_state=seed, stratify=df['accident type']) # accident type
        # df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=seed, stratify=df_temp['accident type'])


        exclude_cols = ['Bert_cause', 'Bert_traffic', 'Tfidf_cause', 'Tfidf_traffic']


        #Seperate X and y
        y1=df_train['accident type']
        X1=df_train.drop(columns=['fatality','accident type', 'accident level', '损伤种类', '损伤位置','环境恶劣', '设备故障'])

        y2=df_test['accident type']
        X2=df_test.drop(columns=['fatality','accident type', 'accident level', '损伤种类', '损伤位置','环境恶劣', '设备故障'])



        categorical_cols = X1.select_dtypes(include=['object']).columns
        categorical_cols = [col for col in categorical_cols if col not in exclude_cols]

        X1 = pd.get_dummies(X1, columns=categorical_cols, drop_first=True)
        expected_columns = X1.columns

        X2 = pd.get_dummies(X2, columns=categorical_cols, drop_first=True)
        X2 = X2.reindex(columns=expected_columns, fill_value=0)

        embedding_model = TextEmbeddingClusterModel(n_clusters=7, seed=seed)
        embedding_model.train(X1)

        # Train first model
        selector, best_model, best_score = cross_validate(seed, X1, y1, scoring='accuracy')  # no conf matrix needed here

        #prepare X2 and X3 for embedding
        tfidf_cause, tfidf_traffic, bert_cause, bert_traffic = embedding_model.predict(X2)
        X2['Tfidf_cause'], X2['Tfidf_traffic'] = tfidf_cause, tfidf_traffic
        X2['Bert_cause'], X2['Bert_traffic'] = bert_cause, bert_traffic

        if selector is not None:
          selector.fit(X1, y1)
          selected_columns = X1.columns[selector.get_support()]
          X1= pd.DataFrame(selector.transform(X1), columns=selected_columns)
          X2= pd.DataFrame(selector.transform(X2), columns=selected_columns)


        y2_pred=best_model.predict(X2)
        y2_acc=accuracy_score(y2_pred,y2)
        conf_matrix = confusion_matrix(y2,y2_pred)

        # Compute macro-averaged precision and recall
        precision_macro = precision_score(y2, y2_pred, average='macro')
        recall_macro = recall_score(y2, y2_pred, average='macro')

        # 7. Store the result
        results.append({
            'seed': seed,
            '1st_train': best_score,
            '1st_test': y2_acc,
            'macro_precision': precision_macro,
            'macro_recall': recall_macro,
            'confusion_matrix': np.array2string(conf_matrix, separator=', ')
        })

    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv, index=False)
    return results_df

seed_pool = [12, 42, 55, 123]
results_df = type_prediction(df, seed_pool)

Started seed 12
Training model: GBM | Feature Selection: False
test average CV score: 0.8110 | Feature Selection: False
Training model: RF | Feature Selection: False
test average CV score: 0.7872 | Feature Selection: False
Training model: GBM | Feature Selection: True
test average CV score: 0.8018 | Feature Selection: True
Training model: RF | Feature Selection: True
test average CV score: 0.7798 | Feature Selection: True
Selected model WITHOUT feature selection: test | Score: 0.8110
Started seed 42
Training model: GBM | Feature Selection: False
test average CV score: 0.8165 | Feature Selection: False
Training model: RF | Feature Selection: False
test average CV score: 0.7963 | Feature Selection: False
Training model: GBM | Feature Selection: True


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [57] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


test average CV score: 0.8128 | Feature Selection: True
Training model: RF | Feature Selection: True


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [57] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


test average CV score: 0.7872 | Feature Selection: True
Selected model WITHOUT feature selection: test | Score: 0.8165
Started seed 55
Training model: GBM | Feature Selection: False
test average CV score: 0.7909 | Feature Selection: False
Training model: RF | Feature Selection: False


KeyboardInterrupt: 